# Compare erosion rates and vegetation lines
This files uses the output from DSAS. It visualises and sumarises the erosion rates and distance calculations for Bull Island in 2017, 2019 and 2021.  

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd

import os
import glob

data_path = "/Users/conorosullivan/Documents/git/COASTAL_MONITORING/sentinel2-irish-coastal-segmentation/data"
figure_path = "/Users/conorosullivan/Google Drive/My Drive/1 UCD/0 research/JP02 - Vegetation Line/figures"

# IBM colour blind palette
blue = "#648fff" #2017
orange = "#fe6100" #2019
red = "#dc267f" #2021
purple = "#785EF0" #overall

## Step 1: process all rates


In [12]:
# Loads transects
transect_path = os.path.join(data_path, "processed/dsas_probalistic/COM_10_0_2021_transects_20251118_111747.geojson")
transect = gpd.read_file(transect_path)
transect.rename(columns={"ObjectID": "TransectId"}, inplace=True)

# Create base table
trasect_base = transect[['TransectId']]

print(len(trasect_base), "transects found")
trasect_base.head()

187 transects found


,TransectId
0,1
1,2
2,3
3,4
4,5


In [14]:
# Distances comparision between ADVLs and MAVLs
distance_paths = glob.glob(os.path.join(data_path, "processed/dsas_probalistic/COM*rates*.geojson"))
distance_paths.sort()
print(len(distance_paths), "COM rate files found")

distance_paths_1 = [path for path in distance_paths if "_1_" in path]
print(len(distance_paths_1), "1 m COM rate files found")

distance_paths_10 = [path for path in distance_paths if "_10_" in path]
print(len(distance_paths_10), "10 m COM rate files found")

distance_paths_20 = [path for path in distance_paths if "_20_" in path]
print(len(distance_paths_20), "20 m COM rate files found")

9 COM rate files found
3 1 m COM rate files found
3 10 m COM rate files found
3 20 m COM rate files found


In [13]:
def get_rates(metric, rate_paths,trasect_base):

    """Format dsas rates into a single table"""

    rates = trasect_base.copy()

    for rate_path in rate_paths:

        file_name = os.path.basename(rate_path)

        rate_id = file_name[:14]
        if "COM" in rate_id:
            rate_id = file_name.split("_")[3] #only use year for distance files
        print(f"Processing {rate_id}")

        period_rates = gpd.read_file(rate_path)
        period_rates = period_rates[['TransectId',metric]]
        period_rates.rename(columns={metric: rate_id}, inplace=True)

        # Left Join rates to transect
        rates = rates.merge(period_rates, on="TransectId", how="left")

    return rates


In [ ]:
# Erosion rates
#epr_erosion_rates = get_rates("EPR", rate_paths, trasect_base)
#epr_erosion_rates.to_csv(os.path.join(data_path, f"results/erosion_rates_EPR.csv"), index=False)
#epr_erosion_rates.head()

Processing ADVL_2017_2019
Processing ADVL_2017_2021
Processing ADVL_2019_2021
Processing MAVL_2017_2019
Processing MAVL_2017_2021
Processing MAVL_2019_2021


,TransectId,ADVL_2017_2019,ADVL_2017_2021,ADVL_2019_2021,MAVL_2017_2019,MAVL_2017_2021,MAVL_2019_2021
0,1,2.70,2.02,1.30,3.58,2.82,2.01
1,2,2.41,1.65,0.85,4.23,3.20,2.11
2,3,3.38,2.16,0.86,8.84,5.49,1.94
3,4,4.73,2.87,0.90,6.09,2.80,-0.68
4,5,5.20,3.20,1.08,6.65,3.22,-0.41


In [15]:
# Erosion rates
NSM_distance_1 = get_rates("NSM", distance_paths_1, trasect_base)
NSM_distance_1.to_csv(os.path.join(data_path, f"results/vl_probalistic_distances_NSM_1.csv"), index=False)

NSM_distance_10 = get_rates("NSM", distance_paths_10, trasect_base)
NSM_distance_10.to_csv(os.path.join(data_path, f"results/vl_probalistic_distances_NSM_10.csv"), index=False)
NSM_distance_10.head()

NSM_distance_20 = get_rates("NSM", distance_paths_20, trasect_base)
NSM_distance_20.to_csv(os.path.join(data_path, f"results/vl_probalistic_distances_NSM_20.csv"), index=False)
NSM_distance_20.head()

Processing 2017
Processing 2019
Processing 2021
Processing 2017
Processing 2019
Processing 2021
Processing 2017
Processing 2019
Processing 2021


,TransectId,2017,2019,2021
0,1,1.15,0.58,-2.84
1,2,0.79,-0.72,-3.97
2,3,7.68,-1.59,-3.79
3,4,0.42,-2.25,0.13
4,5,0.52,-1.43,0.43


## Step 3: Comparison of Lines

In [25]:
import os
import numpy as np
import pandas as pd

# -----------------------------
# Load all distance files
# -----------------------------
resolutions = [1, 10, 20]
distances = {}

for r in resolutions:
    df = pd.read_csv(os.path.join(data_path, f"results/vl_probalistic_distances_NSM_{r}.csv"))
    df.columns = ["TransectId", "2017", "2019", "2021"]
    distances[r] = df

# -----------------------------
# Metric storage
# -----------------------------
metrics = {
    r: {"rmse": [], "mae": []}
    for r in resolutions
}
n_values = []

# -----------------------------
# Per-year calculations
# -----------------------------
years = ["2017", "2019", "2021"]

for year in years:
    # n = number of valid rows in ANY resolution (they share NaN structure)
    n = distances[1][year].notna().sum()
    n_values.append(n)

    for r in resolutions:
        vals = distances[r][year].dropna().values
        rmse = np.sqrt(np.mean(vals ** 2))
        mae = np.mean(np.abs(vals))

        metrics[r]["rmse"].append(rmse)
        metrics[r]["mae"].append(mae)

# -----------------------------
# Overall calculations
# -----------------------------
for r in resolutions:
    all_vals = distances[r].iloc[:, 1:].values.flatten()
    all_vals = all_vals[~np.isnan(all_vals)]
    n_all = len(all_vals)

    rmse = np.sqrt(np.mean(all_vals ** 2))
    mae = np.mean(np.abs(all_vals))

    metrics[r]["rmse"].append(rmse)
    metrics[r]["mae"].append(mae)

# Add overall n (same for all resolutions)
n_values.append(n_all)

# -----------------------------
# Create results table
# -----------------------------
results = pd.DataFrame({
    "Year": [2017, 2019, 2021, "Overall"],
    "No. Transects": n_values,
    "No. S2 Scenes": [6, 11, 16, 6 + 11 + 16],
    "RMSE_1": metrics[1]["rmse"],
    "MAE_1": metrics[1]["mae"],
    "RMSE_10": metrics[10]["rmse"],
    "MAE_10": metrics[10]["mae"],
    "RMSE_20": metrics[20]["rmse"],
    "MAE_20": metrics[20]["mae"],
})

print(results.to_latex(index=False, float_format="%.1f"))
results


\begin{tabular}{lrrrrrrrr}
\toprule
Year & No. Transects & No. S2 Scenes & RMSE_1 & MAE_1 & RMSE_10 & MAE_10 & RMSE_20 & MAE_20 \\
\midrule
2017 & 181 & 6 & 10.5 & 9.8 & 3.4 & 2.7 & 3.4 & 2.5 \\
2019 & 184 & 11 & 9.4 & 8.9 & 2.6 & 1.6 & 2.7 & 1.7 \\
2021 & 186 & 16 & 9.1 & 8.6 & 1.9 & 1.4 & 2.0 & 1.5 \\
Overall & 551 & 33 & 9.6 & 9.1 & 2.7 & 1.9 & 2.8 & 1.9 \\
\bottomrule
\end{tabular}



,Year,No. Transects,No. S2 Scenes,RMSE_1,MAE_1,RMSE_10,MAE_10,RMSE_20,MAE_20
0,2017,181,6,10.470747,9.810276,3.443585,2.670552,3.389179,2.548619
1,2019,184,11,9.373928,8.941304,2.642034,1.637935,2.743576,1.715598
2,2021,186,16,9.067025,8.647634,1.855996,1.434301,1.975491,1.527419
3,Overall,551,33,9.649353,9.127623,2.718308,1.908403,2.757582,1.925717


In [26]:
results_interpolation = pd.read_csv(os.path.join(data_path, f"results/vl_interpolation_error_metrics.csv"))
results_interpolation

,Year,No. Transects,No. S2 Scenes,RMSE,MAE
0,2017,181,6,3.723901,2.793370
1,2019,184,11,2.932976,2.069076
2,2021,186,16,2.575251,1.886989
3,Overall,551,33,3.109139,2.245535


In [ ]:
# Combine MAE of different approaches
mae_combined = pd.DataFrame({
    "Year": results["Year"],
    "No. S2 Scenes": results["No. S2 Scenes"],
    "Linear Interpolation": results_interpolation["MAE"],
    "10 m": results["MAE_1"],
    "1 m": results["MAE_10"],
    "0.5 m": results["MAE_20"],
})
print(mae_combined.to_latex(index=False, float_format="%.1f"))
mae_combined


\begin{tabular}{lrrrrr}
\toprule
Year & No. S2 Scenes & Linear Interpolation & 10 m & 1 m & 0.5 m \\
\midrule
2017 & 6 & 2.8 & 9.8 & 2.7 & 2.5 \\
2019 & 11 & 2.1 & 8.9 & 1.6 & 1.7 \\
2021 & 16 & 1.9 & 8.6 & 1.4 & 1.5 \\
Overall & 33 & 2.2 & 9.1 & 1.9 & 1.9 \\
\bottomrule
\end{tabular}



,Year,No. S2 Scenes,Linear Interpolation,10 m,1 m,0.5 m
0,2017,6,2.793370,9.810276,2.670552,2.548619
1,2019,11,2.069076,8.941304,1.637935,1.715598
2,2021,16,1.886989,8.647634,1.434301,1.527419
3,Overall,33,2.245535,9.127623,1.908403,1.925717


In [34]:
import numpy as np
import pandas as pd

# Create combined MAE table
mae_combined = pd.DataFrame({
    "Year": results["Year"],
    "No. S2 Scenes": results["No. S2 Scenes"],
    "Linear\nInterpolation": results_interpolation["MAE"],
    "10 m": results["MAE_1"],
    "1 m": results["MAE_10"],
    "0.5 m": results["MAE_20"],
})

# Identify the numeric columns
value_cols = ["Linear\nInterpolation", "10 m", "1 m", "0.5 m"]

# Bold the minimum in each row
def bold_min(row):
    vals = row[value_cols].astype(float)
    min_val = vals.min()
    for col in value_cols:
        if np.isclose(row[col], min_val):
            row[col] = f"\\textbf{{{row[col]:.1f}}}"
        else:
            row[col] = f"{row[col]:.1f}"
    return row

mae_formatted = mae_combined.copy().apply(bold_min, axis=1)

# Keys with newline
lin_key = "Linear\nInterpolation"

# Build LaTeX table
latex = r"""
\begin{table}[ht]
\centering
\caption{True and estimated classes, with the sample mean.}\label{tab:MyTable}
\begin{tabular}{l r r r r r}
\toprule
 & & & \multicolumn{3}{c}{Upsampled Resolution} \\
\cmidrule(lr){4-6}
Year & No. S2 Scenes & \begin{tabular}{c} Linear \\ Interpolation \end{tabular} & 10 m & 1 m & 0.5 m \\
\midrule
"""

for i, row in mae_formatted.iterrows():

    if row["Year"] == "Overall":
        latex += r"\midrule" + "\n"

    year = row["Year"]
    s2 = row["No. S2 Scenes"]
    lin = row[lin_key]
    m10 = row["10 m"]
    m1 = row["1 m"]
    m05 = row["0.5 m"]

    latex += f"{year} & {s2} & {lin} & {m10} & {m1} & {m05} \\\\\n"

latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print(latex)




\begin{table}[ht]
\centering
\caption{True and estimated classes, with the sample mean.}\label{tab:MyTable}
\begin{tabular}{l r r r r r}
\toprule
 & & & \multicolumn{3}{c}{Upsampled Resolution} \\
\cmidrule(lr){4-6}
Year & No. S2 Scenes & \begin{tabular}{c} Linear \\ Interpolation \end{tabular} & 10 m & 1 m & 0.5 m \\
\midrule
2017 & 6 & 2.8 & 9.8 & 2.7 & \textbf{2.5} \\
2019 & 11 & 2.1 & 8.9 & \textbf{1.6} & 1.7 \\
2021 & 16 & 1.9 & 8.6 & \textbf{1.4} & 1.5 \\
\midrule
Overall & 33 & 2.2 & 9.1 & \textbf{1.9} & 1.9 \\
\bottomrule
\end{tabular}
\end{table}

